In [1]:
import numpy as np
import xarray as xr
from glob import glob
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
dsList = []

for year in range(2014, 2025):
    print(year)
    folder = f'/proj/cmip6/data/ocean_reanalysis/glorys12v1/{year:04d}/'
    ds = xr.open_mfdataset(glob(folder + '*.nc'))
    
    sst = ds.thetao.isel(depth=0)
    sst = sst.sel(latitude = 0, method='nearest')
    
    uo = ds.uo.isel(depth=0)
    uo = uo.sel(latitude = 0, method='nearest')
    
    zos = ds.zos.sel(latitude = 0, method='nearest')
    
    mlotst = ds.mlotst.sel(latitude =0, method='nearest')
    
    ds.close()
    
    subds = xr.Dataset()
    subds['uo'] = uo
    subds['sst'] = sst
    subds['zos'] = zos
    subds['mld'] = mlotst
    
    dsList.append(subds)
    

2014


OSError: no files to open

In [ ]:
fullDS = xr.concat(dsList, dim='time').compute()

In [ ]:
fullDS

In [ ]:
writeFname = '../../WPWP_GLORYS_data/equator_sst_u_ssh_mld_2014_2024.nc'
new_lon = (fullDS['longitude'].values + 360) % 360
fullDS = fullDS.assign_coords(longitude=new_lon)
fullDS = fullDS.sortby('longitude')
fullDS = fullDS.sel(longitude=slice(106, 296))

# Write immediately
fullDS.to_netcdf(writeFname, unlimited_dims='time')

In [ ]:
fullDS